### **DimUser**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
import importlib
project_path=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
import utils.transformations as transformations
importlib.reload(transformations)
reusable = transformations.reusable

 

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimUser/") 
df.display()

##**autoloader**


In [0]:
df_user = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format","parquet")
                .option("cloudFiles.schemaLocation","abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimUser/checkpoint")
                .load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimUser/"))


In [0]:
display(df,
    checkpointLocation="abfss://bronze@<storage-account>.dfs.core.windows.net/checkpoint/")

In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])

In [0]:
df_user = df_user.dropDuplicates(subset = ["user_id"])

In [0]:
(df_user.writeStream.format("delta")
                    .option("outputMode","append")
                    .option("checkpointLocation","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimUser/checkpoint")
                    .trigger(once=True)
                    .option("path","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimUser/data")
                    .toTable("spotify_catalog.silver.DimUser"))


## **Dimartist**

In [0]:
df_art = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format","parquet")
                .option("cloudFiles.schemaLocation","abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimArtist/checkpoint")
                .load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimArtist/"))

In [0]:
df_art_obj = reusable()
df_art = df_art_obj.dropColumns(df_art, ['_rescued_data'])

In [0]:
df_art = df_art.dropDuplicates(subset = ["artist_id"])

In [0]:
 (df_art.writeStream.format("delta")
                    .option("outputMode","append")
                    .option("checkpointLocation","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimArtist/checkpoint")
                    .trigger(once=True)
                    .option("path","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimArtist/data")
                    .toTable("spotify_catalog.silver.DimArtist"))

### **DimTrack**

In [0]:
df_track = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format","parquet")
                .option("cloudFiles.schemaLocation","abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimTrack/checkpoint")
                .load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimTrack/"))

In [0]:
df_track_obj =reusable()
df_track =df_track_obj.dropColumns(df_track,['_rescued_data'])

In [0]:
df_track =df_track.withColumn("duration_flag",when(col("duration_sec")<150,"low").when(col("duration_sec")<300,"medium").otherwise("high"))

In [0]:
df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))

In [0]:
(df_track.writeStream.format("delta")
                    .option("outputMode","append")
                    .option("checkpointLocation","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimTrack/checkpoint")
                    .trigger(once=True)
                    .option("path","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimTrack/data")
                    .toTable("spotify_catalog.silver.DimTrack"))

## **DimDate**


In [0]:
df_date = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format","parquet")
                .option("cloudFiles.schemaLocation","abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimDate/checkpoint")
                .load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/DimDate/"))

In [0]:
df_date_obj  = reusable()
df_date = df_date_obj.dropColumns(df_date,["_rescued_data"])

In [0]:
(df_date.writeStream.format("delta")
                    .option("outputMode","append")
                    .option("checkpointLocation","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimDate/checkpoint")
                    .trigger(once=True)
                    .option("path","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/DimDate/data")
                    .toTable("spotify_catalog.silver.DimDate"))

## **FactStream**

In [0]:
df_fact = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format","parquet")
                .option("cloudFiles.schemaLocation","abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/FactStream/checkpoint")
                .load("abfss://bronze@storagespotifyprojectsk.dfs.core.windows.net/FactStream/"))

In [0]:
df_factstream_obj  = reusable()
df_fact = df_factstream_obj.dropColumns(df_fact,["_rescued_data"])

In [0]:
(df_fact.writeStream.format("delta")
                    .option("outputMode","append")
                    .option("checkpointLocation","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/FactStream/checkpoint")
                    .trigger(once=True)
                    .option("path","abfss://silver@storagespotifyprojectsk.dfs.core.windows.net/FactStream/data")
                    .toTable("spotify_catalog.silver.FactStream"))